## ML_1015: Scaled Dot-Product Attention

**Difficulty**: Hard | **TorchCode**: #05

### Problem
Implement scaled dot-product attention. Use `torch.bmm` for batched matrix multiplication. Supports cross-attention (Q sequence length ≠ K/V sequence length).

### Formula
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
import torch
import math

def scaled_dot_product_attention(Q, K, V):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
import torch
import math

# --- Test 1: Output shape ---
torch.manual_seed(42)
B, S, D = 2, 4, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out = scaled_dot_product_attention(Q, K, V)
assert out.shape == (B, S, D)

# --- Test 2: Numerical correctness ---
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out, ref, atol=1e-5)

# --- Test 3: Gradient check ---
Q = torch.randn(2, 4, 8, requires_grad=True)
K = torch.randn(2, 4, 8, requires_grad=True)
V = torch.randn(2, 4, 8, requires_grad=True)
scaled_dot_product_attention(Q, K, V).sum().backward()
assert Q.grad is not None and K.grad is not None and V.grad is not None

# --- Test 4: Cross-attention (seq_q != seq_k) ---
Q2 = torch.randn(1, 3, 16); K2 = torch.randn(1, 5, 16); V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
assert out2.shape == (1, 3, 32)

print('All tests passed!')